# Storage & DuckDB Queries

This notebook explores the **storage layer** of Equity Lake:

- `EquityDataDB` — DuckDB context manager for analytical queries
- Pre-built `QueryExamples` — 8 analytical queries ready to run
- Storage size breakdown by market
- Delta Lake: version history and time-travel queries

**Prerequisites:** Ingested data in `data/lake/` (run `uv run equity ingest`).

In [1]:
import sys
sys.path.insert(0, "../src")

from equity_lake.storage.duckdb import EquityDataDB, QueryExamples
from equity_lake.core.paths import LAKE_DIR
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

print(f"Lake directory: {LAKE_DIR}")
print(f"Lake exists: {LAKE_DIR.exists()}")

Lake directory: /Users/minghao/Desktop/personal/equity_lake/data/lake
Lake exists: True


## 1. EquityDataDB — Quick Start

`EquityDataDB` is a DuckDB context manager that auto-creates views for all markets.

In [2]:
with EquityDataDB() as db:
    # List available views
    views = db.query("SHOW TABLES")
    print("Available views:")
    for v in views["name"].tolist():
        print(f"  - {v}")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Available views:
  - cn_ashare
  - equity_all
  - hk_sg_equity
  - us_equity


## 2. Latest Data Summary

Query 1: Overview of latest data per market.

In [3]:
with EquityDataDB() as db:
    examples = QueryExamples(db)
    summary = db.query(examples.query_1_latest_data_summary)
    print(f"Rows: {len(summary)}")
summary

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Invalid Input Error: Please provide either a DuckDBPyStatement or a string representing the query


Rows: 0


""


## 3. Top Volume Stocks

Query 2: Which stocks have the highest trading volume?

In [4]:
with EquityDataDB() as db:
    examples = QueryExamples(db)
    top_vol = db.query(examples.query_2_top_volume_stocks)

if not top_vol.empty:
    fig = px.bar(
        top_vol.head(20),
        x="ticker",
        y="total_volume",
        title="Top 20 Stocks by Trading Volume",
        labels={"ticker": "Ticker", "total_volume": "Total Volume"},
        color="total_volume",
        color_continuous_scale="Viridis",
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print("No volume data available")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Invalid Input Error: Please provide either a DuckDBPyStatement or a string representing the query


No volume data available


## 4. Top Gainers & Losers

Query 3: Biggest daily price movers.

In [5]:
with EquityDataDB() as db:
    examples = QueryExamples(db)
    movers = db.query(examples.query_3_top_gainers_losers)

if not movers.empty:
    fig = px.bar(
        movers.head(30),
        x="ticker",
        y="daily_return",
        color="daily_return",
        color_continuous_scale="RdYlGn",
        title="Top Gainers & Losers (Daily Return %)",
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print("No return data available")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Invalid Input Error: Please provide either a DuckDBPyStatement or a string representing the query


No return data available


## 5. Moving Averages

Query 5: SMA overlay on price data.

In [6]:
with EquityDataDB() as db:
    examples = QueryExamples(db)
    ma_df = db.query(examples.query_5_moving_averages)

if not ma_df.empty:
    ticker_sample = ma_df["ticker"].unique()[0] if "ticker" in ma_df.columns else None
    if ticker_sample:
        ticker_data = ma_df[ma_df["ticker"] == ticker_sample].tail(120)
        fig = go.Figure()
        if "close" in ticker_data.columns:
            fig.add_trace(go.Scatter(x=ticker_data["date"], y=ticker_data["close"], name="Close", line=dict(width=1)))
        for col in ticker_data.columns:
            if "sma" in col.lower() or "ma" in col.lower():
                fig.add_trace(go.Scatter(x=ticker_data["date"], y=ticker_data[col], name=col.upper(), line=dict(dash="dash")))
        fig.update_layout(title=f"Moving Averages — {ticker_sample}", height=500, xaxis_title="Date", yaxis_title="Price")
        fig.show()
    else:
        print(ma_df.head())
else:
    print("No moving average data available")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Invalid Input Error: Please provide either a DuckDBPyStatement or a string representing the query


No moving average data available


## 6. Volatility Analysis

Query 6: Volatility across stocks.

In [7]:
with EquityDataDB() as db:
    examples = QueryExamples(db)
    vol_df = db.query(examples.query_6_volatility_analysis)

if not vol_df.empty:
    fig = px.scatter(
        vol_df,
        x="ticker",
        y=vol_df.columns[-1] if len(vol_df.columns) > 1 else vol_df.columns[0],
        title="Volatility Analysis by Ticker",
        color=vol_df.columns[-1] if len(vol_df.columns) > 1 else vol_df.columns[0],
        color_continuous_scale="Plasma",
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print("No volatility data available")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Invalid Input Error: Please provide either a DuckDBPyStatement or a string representing the query


No volatility data available


## 7. Custom Queries

Run your own SQL against the unified `equity_all` view.

In [8]:
with EquityDataDB() as db:
    # Cross-market comparison
    custom_sql = '''
    SELECT
        SUBSTR(_dir, 1, POSITION('/date=' IN _dir) - 1) as market_path,
        COUNT(DISTINCT ticker) as num_tickers,
        COUNT(*) as total_rows,
        MIN(date) as min_date,
        MAX(date) as max_date
    FROM equity_all
    GROUP BY 1
    ORDER BY total_rows DESC
    '''
    try:
        result = db.query(custom_sql)
        print(result.to_string(index=False))
    except Exception as e:
        print(f"Query error: {e}")
        print("Trying simpler query...")
        result = db.query("SELECT COUNT(*) as cnt FROM equity_all")
        print(f"Total rows in equity_all: {result['cnt'].iloc[0]}")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Binder Error: Referenced column "_dir" not found in FROM clause!
Candidate bindings: "ticker", "date", "high"

LINE 3:         SUBSTR(_dir, 1, POSITION('/date=' IN _dir) - 1) as market_path,
                       ^


Empty DataFrame
Columns: []
Index: []


## 8. Storage Size Breakdown

Check how much disk space each market uses.

In [9]:
import os

size_data = []
for market_dir in LAKE_DIR.iterdir():
    if market_dir.is_dir():
        total_size = sum(f.stat().st_size for f in market_dir.rglob("*") if f.is_file())
        num_files = sum(1 for f in market_dir.rglob("*") if f.is_file())
        if total_size > 0:
            size_data.append({
                "Market": market_dir.name,
                "Size (MB)": round(total_size / 1024 / 1024, 2),
                "Files": num_files,
            })

if size_data:
    size_df = pd.DataFrame(size_data).sort_values("Size (MB)", ascending=False)
    print(size_df.to_string(index=False))

    fig = px.pie(size_df, values="Size (MB)", names="Market", title="Storage Size by Market")
    fig.update_layout(height=500)
    fig.show()
else:
    print("No data in lake directory — run ingestion first")

          Market  Size (MB)  Files
       us_equity      59.61   6698
    hk_sg_equity      53.18   6431
       cn_ashare      47.63   6093
        features       0.61     51
         us_news       0.19      6
macro_indicators       0.00      1


## 9. Delta Lake — Version History

If Delta Lake is enabled, inspect table versions and time-travel.

In [10]:
from equity_lake.storage.delta import delta_table_version, read_delta

for market in ["us_equity", "cn_ashare"]:
    delta_path = LAKE_DIR / market
    if delta_path.exists() and (delta_path / "_delta_log").exists():
        try:
            version = delta_table_version(str(delta_path))
            print(f"{market}: Delta version {version}")
        except Exception as e:
            print(f"{market}: {e}")
    else:
        print(f"{market}: Not a Delta table")

print("\nTo migrate Parquet to Delta:")
print("  dotenvx run -- uv run equity delta-migrate --market us")
print("\nTo compact Delta files:")
print("  dotenvx run -- uv run equity delta-compact --market us")

us_equity: Delta version 2600
cn_ashare: Delta version 2426

To migrate Parquet to Delta:
  dotenvx run -- uv run equity delta-migrate --market us

To compact Delta files:
  dotenvx run -- uv run equity delta-compact --market us


## Next Steps

- **[04-hamilton-features.ipynb](04-hamilton-features.ipynb)** — Generate features from stored data
- **[08-backtesting.ipynb](08-backtesting.ipynb)** — Run backtests on stored data
- **[10-validation-and-quality.ipynb](10-validation-and-quality.ipynb)** — Validate data quality